# Stitch Bot — Colab

Apri questo notebook dal telefono (o dal PC) e premi **Runtime → Run all**.
Gira sui server di Google: il telefono serve solo per aprirlo e guardarlo.

**Cosa fa già adesso:** lo *splitter* delle reference (una reference lunga → 3 fette).

**Cosa manca:** la parte che manda a Stitch (`send_to_stitch.py`), che va caricata
intatta su Drive — vedi l'ultima sezione.

## 1) Collega Google Drive
Ti chiederà il permesso: accetta. Serve per leggere le reference e salvare le fette.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 2) Installa le librerie

In [ ]:
!pip -q install Pillow

## 3) Splitter delle reference
Imposta **SRC** (cartella con le reference su Drive) e **OUT** (dove salvare le fette),
poi esegui. Ogni reference finisce in una sottocartella con `parte-1-di-3.png`, ecc.
La HERO è sempre nella `parte-1`.

In [ ]:
import re
from pathlib import Path
from PIL import Image

# >>> IMPOSTA QUESTI DUE PERCORSI (dentro il tuo Drive) <<<
SRC = '/content/drive/MyDrive/REFENCE'                 # cartella con le reference
OUT = '/content/drive/MyDrive/references_split'        # dove salvare le fette
N   = 3                                                # in quante parti (default 3)

IMG_EXT = {'.png', '.jpg', '.jpeg', '.webp'}

def slug(name):
    s = re.sub(r'\.(png|jpe?g|webp)$', '', name, flags=re.I)
    s = re.sub(r'[^a-z0-9]+', '-', s, flags=re.I).strip('-').lower()
    return s[:60] or 'ref'

src, out = Path(SRC), Path(OUT)
assert src.is_dir(), f'Cartella reference non trovata: {src}'
files = sorted(p for p in src.iterdir() if p.is_file() and p.suffix.lower() in IMG_EXT)
assert files, f'Nessuna immagine in {src}'
out.mkdir(parents=True, exist_ok=True)

for p in files:
    im = Image.open(p).convert('RGB')
    W, H = im.size
    band = -(-H // N)          # ceil(H/N)
    ov = round(H * 0.02)       # 2% overlap ai bordi
    sub = out / slug(p.name)
    sub.mkdir(parents=True, exist_ok=True)
    for old in sub.glob('parte-*.png'):
        old.unlink()
    for i in range(N):
        y0 = max(0, i * band - (ov if i > 0 else 0))
        y1 = min(H, (i + 1) * band + (ov if i < N - 1 else 0))
        im.crop((0, y0, W, y1)).save(sub / f'parte-{i+1}-di-{N}.png', compress_level=1)
    print('ok', slug(p.name), f'({W}x{H}, {N} pezzi)')

print('\nFATTO:', len(files), 'reference divise in', out)

## 4) Controllo veloce delle fette prodotte

In [ ]:
from pathlib import Path
for sub in sorted(Path(OUT).iterdir()):
    if sub.is_dir():
        pezzi = sorted(sub.glob('parte-*.png'))
        print(sub.name, '->', len(pezzi), 'pezzi')

## 5) 🔒 SIGILLATO — invio a Stitch (`send_to_stitch.py`)

Questa è la tua automazione, da **NON modificare**. Caricala su Drive intatta,
insieme ai suoi file (i 5 prompt, gli helper), poi imposta qui sotto la cartella.

> ⚠️ Nota onesta: `send_to_stitch.py` è scritto per **macOS** (usa `osascript`,
> `pbcopy`, permessi di Accessibilità). Su Colab (Linux) l'avvio va adattato a
> Playwright puro — senza toccare la logica di prompt/controllo/invio. Serve anche
> gestire il **login Google** (bloccato dagli IP dei datacenter): di norma si fa il
> login una volta e si riusa il profilo del browser.

In [ ]:
# Prepara il browser per l'invio (serve quando send_to_stitch.py sarà presente)
!pip -q install playwright
!python -m playwright install chromium

In [ ]:
from pathlib import Path

# >>> cartella su Drive dove hai caricato send_to_stitch.py e i suoi file <<<
SENDER_DIR = '/content/drive/MyDrive/reference_splitter'

script = Path(SENDER_DIR) / 'send_to_stitch.py'
if script.exists():
    print('Trovato:', script)
    print('Pronto per il porting/avvio su Colab.')
else:
    print('MANCANTE: send_to_stitch.py non trovato in', SENDER_DIR)
    print('Caricalo su Drive (intatto) e riesegui questa cella.')